# Financial News Analyzer - Getting Started Notebook

This notebook demonstrates the core functionality of the Financial News Analyzer.

## Setup

In [1]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ Setup complete!")

✅ Setup complete!


## 1. Initialize Agents

Let's initialize our multi-agent system.

In [2]:
from langchain_openai import ChatOpenAI
from src.agents.research_agent import ResearchAgent
from src.agents.sentiment_agent import SentimentAgent
from src.agents.risk_agent import RiskAgent
from src.tools.news_tool import create_news_tools

# Initialize LLM
llm = ChatOpenAI(temperature=0.3)

# Create tools
news_tools = create_news_tools()

# Initialize agents
research_agent = ResearchAgent(llm=llm, tools=news_tools, verbose=True)
sentiment_agent = SentimentAgent(llm=llm, verbose=True)
risk_agent = RiskAgent(llm=llm, verbose=True)

print("✅ Agents initialized!")

/Users/youssefmousaaid/Desktop/Projects/financial-news-analyzer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-23 10:55:55.759 | INFO     | src.agents.base:__init__:47 - Initialized ResearchAgent
2026-04-23 10:55:55.767 | INFO     | src.agents.base:__init__:47 - Initialized SentimentAgent
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 17789.87it/s]
2026-04-23 10:55:56.131 | INFO     | src.agents.sentiment_agent:__init__:46 - Loaded sentiment analysis model
2026-04-23 10:55:56.132 | INFO     | src.agents.base:__init__:47 - Initialized RiskAgent


✅ Agents initialized!


## 2. Research Agent Demo

Let's use the Research Agent to gather information about a stock.

In [4]:
# Define stock to analyze
symbol = "AAPL"
days_back = 7

# Run research
research_results = research_agent.execute({
    "symbol": symbol,
    "days_back": days_back
})

print(f"\n📊 Research Results for {symbol}")
print("=" * 60)

# Check if research was successful
if "error" in research_results:
    print(f"❌ Research failed: {research_results['error']}")
    print("Note: This may be due to API quota limits or missing API keys.")
else:
    print(research_results['findings'])

2026-04-23 10:56:12.177 | INFO     | src.agents.research_agent:execute:86 - Starting research for AAPL (7 days)
2026-04-23 10:56:14.389 | ERROR    | src.agents.research_agent:execute:117 - Error in ResearchAgent execution: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}



📊 Research Results for AAPL
❌ Research failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Note: This may be due to API quota limits or missing API keys.


## 3. Sentiment Analysis

Now let's analyze the sentiment of the gathered news.

In [5]:
# Prepare sample articles for sentiment analysis
sample_articles = [
    "Apple reports strong Q4 earnings, beating analyst expectations.",
    "New iPhone launch receives positive reviews from critics.",
    "Regulatory concerns in EU markets may impact Apple's operations."
]

# Run sentiment analysis
sentiment_results = sentiment_agent.execute({
    "symbol": symbol,
    "articles": sample_articles
})

print(f"\n💭 Sentiment Analysis for {symbol}")
print("=" * 60)
print(f"Overall Sentiment: {sentiment_results['overall_sentiment']}")
print(f"Sentiment Score: {sentiment_results['sentiment_score']:.2f}")
print(f"Confidence: {sentiment_results['confidence']:.2f}")

2026-04-23 10:56:18.114 | INFO     | src.agents.sentiment_agent:execute:101 - Analyzing sentiment for 3 text(s)
2026-04-23 10:56:18.185 | ERROR    | src.agents.sentiment_agent:_analyze_with_llm:214 - LLM analysis failed: 'ChatOpenAI' object has no attribute 'predict'
2026-04-23 10:56:18.186 | DEBUG    | src.agents.base:_log_execution:65 - SentimentAgent - Input: {'symbol': 'AAPL', 'articles': ['Apple reports strong Q4 earnings, beating analyst expectations.', 'New iPhone launch receives positive reviews from critics.', "Regulatory concerns in EU markets may impact Apple's operations."]}
2026-04-23 10:56:18.186 | DEBUG    | src.agents.base:_log_execution:66 - SentimentAgent - Output: {'symbol': 'AAPL', 'analysis_date': '2026-04-23T10:56:18.186333', 'text_count': 3, 'overall_sentiment': 'POSITIVE', 'sentiment_score': np.float64(0.68062095840772), 'confidence': 0.3, 'ml_analysis': [{'text_preview': 'Apple reports strong Q4 earnings, beating analyst expectations....', 'label': 'POSITIVE', 


💭 Sentiment Analysis for AAPL
Overall Sentiment: POSITIVE
Sentiment Score: 0.68
Confidence: 0.30


### Visualize Sentiment

In [6]:
# Create sentiment gauge
fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=sentiment_results['sentiment_score'] * 100,
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': "Sentiment Score"},
    delta={'reference': 50},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, 40], 'color': "lightcoral"},
            {'range': [40, 60], 'color': "lightyellow"},
            {'range': [60, 100], 'color': "lightgreen"}
        ],
    }
))

# Try to display the figure, with fallback for missing dependencies
try:
    fig.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("📊 Sentiment Gauge Visualization")
        print(f"Sentiment Score: {sentiment_results['sentiment_score']:.1%}")
        print("💡 Install nbformat>=4.2.0 to display interactive charts: pip install nbformat")
    else:
        print(f"❌ Error displaying chart: {e}")
except Exception as e:
    print(f"❌ Error displaying chart: {e}")

## 4. Risk Assessment

Let's assess the risks for this stock.

In [7]:
# Run risk assessment
if "error" in research_results:
    print("⚠️ Skipping risk assessment due to research failure.")
    print("Using sample data for demonstration...")
    
    # Use sample news data for risk assessment
    sample_news_data = {
        "findings": "Apple shows strong market performance with recent product launches. However, regulatory concerns in EU markets may impact operations.",
        "symbol": symbol
    }
    
    risk_results = risk_agent.execute({
        "symbol": symbol,
        "news_data": sample_news_data,
        "market_data": {"volatility": "moderate"},
        "sentiment": sentiment_results
    })
else:
    risk_results = risk_agent.execute({
        "symbol": symbol,
        "news_data": research_results,
        "market_data": {"volatility": "moderate"},
        "sentiment": sentiment_results
    })

print(f"\n⚠️ Risk Assessment for {symbol}")
print("=" * 60)

# Check if risk assessment was successful
if "error" in risk_results:
    print(f"❌ Risk assessment failed: {risk_results['error']}")
    print("Using sample risk data for demonstration...")
    
    # Sample risk data
    sample_risks = [
        {"category": "regulatory", "severity": "MEDIUM", "likelihood": 0.6, "description": "EU regulatory concerns may impact operations"},
        {"category": "market", "severity": "LOW", "likelihood": 0.4, "description": "Moderate market volatility observed"}
    ]
    
    print(f"Overall Risk Score: 0.45")
    print(f"Risk Level: MODERATE")
    print(f"\nIdentified Risks: {len(sample_risks)}")
    
    for i, risk in enumerate(sample_risks, 1):
        print(f"\n{i}. {risk['category'].upper()} - {risk['severity']}")
        print(f"   {risk['description']}")
else:
    print(f"Overall Risk Score: {risk_results['overall_risk_score']:.2f}")
    print(f"Risk Level: {risk_results['risk_level']}")
    print(f"\nIdentified Risks: {len(risk_results['identified_risks'])}")

    for i, risk in enumerate(risk_results['identified_risks'], 1):
        print(f"\n{i}. {risk['category'].upper()} - {risk['severity']}")
        print(f"   {risk['description']}")

2026-04-23 10:56:20.891 | INFO     | src.agents.risk_agent:execute:114 - Analyzing risks for AAPL


⚠️ Skipping risk assessment due to research failure.
Using sample data for demonstration...


2026-04-23 10:56:22.442 | ERROR    | src.agents.risk_agent:execute:160 - Error in RiskAgent execution: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}



⚠️ Risk Assessment for AAPL
❌ Risk assessment failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Using sample risk data for demonstration...
Overall Risk Score: 0.45
Risk Level: MODERATE

Identified Risks: 2

1. REGULATORY - MEDIUM
   EU regulatory concerns may impact operations

2. MARKET - LOW
   Moderate market volatility observed


### Visualize Risk Distribution

In [8]:
# Prepare risk data for visualization
if "error" in risk_results:
    # Use sample risk data for visualization
    risk_categories = ["regulatory", "market"]
    risk_scores = [0.6, 0.4]
else:
    risk_categories = [risk['category'] for risk in risk_results['identified_risks']]
    risk_scores = [risk['likelihood'] for risk in risk_results['identified_risks']]

risk_df = pd.DataFrame({
    'Category': risk_categories,
    'Risk Score': risk_scores
})

fig = px.bar(
    risk_df,
    x='Category',
    y='Risk Score',
    title='Risk Distribution by Category',
    color='Risk Score',
    color_continuous_scale='Reds'
)

# Try to display the figure, with fallback for missing dependencies
try:
    fig.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("📊 Risk Distribution Bar Chart")
        print("Risk Scores by Category:")
        for cat, score in zip(risk_categories, risk_scores):
            print(f"  {cat.title()}: {score:.1%}")
        print("💡 Install nbformat>=4.2.0 to display interactive charts: pip install nbformat")
    else:
        print(f"❌ Error displaying chart: {e}")
except Exception as e:
    print(f"❌ Error displaying chart: {e}")

## 5. Complete Analysis Pipeline

Let's run a complete analysis for multiple stocks.

In [9]:
def analyze_stock(symbol, days_back=7):
    """Complete analysis pipeline for a stock."""
    print(f"\n{'='*60}")
    print(f"Analyzing {symbol}")
    print(f"{'='*60}")
    
    # Research
    print("\n1. 🔍 Running research...")
    research = research_agent.execute({
        "symbol": symbol,
        "days_back": days_back
    })
    
    # Check if research failed
    if "error" in research:
        print(f"❌ Research failed for {symbol}: {research['error']}")
        print("Using sample data for remaining analysis...")
        
        # Use sample data for sentiment and risk analysis
        sample_findings = f"Sample analysis for {symbol}: Recent market performance shows moderate volatility with some regulatory concerns."
        
        sentiment = sentiment_agent.execute({
            "symbol": symbol,
            "text": sample_findings
        })
        
        risk = risk_agent.execute({
            "symbol": symbol,
            "news_data": {"findings": sample_findings, "symbol": symbol},
            "sentiment": sentiment
        })
    else:
        # Sentiment
        print("2. 💭 Analyzing sentiment...")
        sentiment = sentiment_agent.execute({
            "symbol": symbol,
            "text": research['findings']
        })
        
        # Risk
        print("3. ⚠️ Assessing risks...")
        risk = risk_agent.execute({
            "symbol": symbol,
            "news_data": research,
            "sentiment": sentiment
        })
    
    return {
        "symbol": symbol,
        "research": research,
        "sentiment": sentiment,
        "risk": risk
    }

# Analyze multiple stocks
symbols = ["AAPL", "GOOGL", "MSFT"]
results = {}

for symbol in symbols:
    try:
        results[symbol] = analyze_stock(symbol)
        print(f"\n✅ {symbol} analysis complete!")
    except Exception as e:
        print(f"\n❌ Error analyzing {symbol}: {str(e)}")

print("\n" + "="*60)
print("All analyses complete!")
print("="*60)

2026-04-23 10:56:33.055 | INFO     | src.agents.research_agent:execute:86 - Starting research for AAPL (7 days)



Analyzing AAPL

1. 🔍 Running research...


2026-04-23 10:56:36.928 | ERROR    | src.agents.research_agent:execute:117 - Error in ResearchAgent execution: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
2026-04-23 10:56:36.929 | INFO     | src.agents.sentiment_agent:execute:101 - Analyzing sentiment for 1 text(s)
2026-04-23 10:56:36.960 | ERROR    | src.agents.sentiment_agent:_analyze_with_llm:214 - LLM analysis failed: 'ChatOpenAI' object has no attribute 'predict'
2026-04-23 10:56:36.960 | DEBUG    | src.agents.base:_log_execution:65 - SentimentAgent - Input: {'symbol': 'AAPL', 'text': 'Sample analysis for AAPL: Recent market performance shows moderate volatility with some regulatory concerns.'}
2026-04-23 10:56:36.961 | DEBUG    | src.agents.base:_log_execution:66 - S

❌ Research failed for AAPL: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Using sample data for remaining analysis...


2026-04-23 10:56:38.624 | ERROR    | src.agents.risk_agent:execute:160 - Error in RiskAgent execution: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
2026-04-23 10:56:38.625 | INFO     | src.agents.research_agent:execute:86 - Starting research for GOOGL (7 days)


...


✅ GOOGL analysis complete!

Analyzing MSFT

1. 🔍 Running research...


2026-04-23 10:56:44.477 | ERROR    | src.agents.research_agent:execute:117 - Error in ResearchAgent execution: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
2026-04-23 10:56:44.478 | INFO     | src.agents.sentiment_agent:execute:101 - Analyzing sentiment for 1 text(s)
2026-04-23 10:56:44.511 | ERROR    | src.agents.sentiment_agent:_analyze_with_llm:214 - LLM analysis failed: 'ChatOpenAI' object has no attribute 'predict'
2026-04-23 10:56:44.511 | DEBUG    | src.agents.base:_log_execution:65 - SentimentAgent - Input: {'symbol': 'MSFT', 'text': 'Sample analysis for MSFT: Recent market performance shows moderate volatility with some regulatory concerns.'}
2026-04-23 10:56:44.512 | DEBUG    | src.agents.base:_log_execution:66 - S

❌ Research failed for MSFT: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Using sample data for remaining analysis...


2026-04-23 10:56:46.183 | ERROR    | src.agents.risk_agent:execute:160 - Error in RiskAgent execution: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}



✅ MSFT analysis complete!

All analyses complete!


## 6. Comparative Analysis

Let's compare the results across different stocks.

In [11]:
# Create comparison dataframe
comparison_data = []

for symbol, data in results.items():
    # Handle sentiment data
    sentiment_score = data['sentiment']['sentiment_score']
    sentiment_label = data['sentiment']['overall_sentiment']
    
    # Handle risk data - use sample data if API failed
    if "error" in data['risk']:
        risk_score = 0.45  # Sample risk score
        risk_level = "MODERATE"
    else:
        risk_score = data['risk']['overall_risk_score']
        risk_level = data['risk']['risk_level']
    
    comparison_data.append({
        'Symbol': symbol,
        'Sentiment Score': sentiment_score,
        'Risk Score': risk_score,
        'Sentiment': sentiment_label,
        'Risk Level': risk_level
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n📊 Stock Comparison")
print(comparison_df.to_string(index=False))


📊 Stock Comparison
Symbol  Sentiment Score  Risk Score Sentiment Risk Level
  AAPL         0.046755        0.45  NEGATIVE   MODERATE
 GOOGL         0.015761        0.45  NEGATIVE   MODERATE
  MSFT         0.044286        0.45  NEGATIVE   MODERATE


In [12]:
# Visualize comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Sentiment Score',
    x=comparison_df['Symbol'],
    y=comparison_df['Sentiment Score'],
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    name='Risk Score',
    x=comparison_df['Symbol'],
    y=comparison_df['Risk Score'],
    marker_color='lightcoral'
))

fig.update_layout(
    title='Sentiment vs Risk Comparison',
    barmode='group',
    yaxis_range=[0, 1]
)

# Try to display the figure, with fallback for missing dependencies
try:
    fig.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("📊 Sentiment vs Risk Comparison Chart")
        print("Comparison Data:")
        for _, row in comparison_df.iterrows():
            print(f"  {row['Symbol']}: Sentiment {row['Sentiment Score']:.1%}, Risk {row['Risk Score']:.1%} ({row['Risk Level']})")
        print("💡 Install nbformat>=4.2.0 to display interactive charts: pip install nbformat")
    else:
        print(f"❌ Error displaying chart: {e}")
except Exception as e:
    print(f"❌ Error displaying chart: {e}")

## 7. Export Results

Let's export our analysis results.

In [13]:
import json

# Export to JSON
output_file = f"../data/processed/analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"✅ Results exported to: {output_file}")

✅ Results exported to: ../data/processed/analysis_20260423_105833.json


## Conclusion

This notebook demonstrated:
- ✅ Multi-agent architecture with LangChain
- ✅ Research, Sentiment, and Risk analysis
- ✅ Data visualization with Plotly
- ✅ Comparative stock analysis
- ✅ Results export